# Patching and model behaviour

Greedy whole-number generations under the $\mathbf{u}$, $\mathbf{v}_1$, $\mathbf{v}_2$, plane and L15-DAS interchanges, on the makefig evaluation triples. IIA is scored on the first digit and on the whole number.

$$\mathrm{IIA}_{\text{first}} = \mathbb{1}\left[\arg\max_t \ell_{\text{patched},t} = \text{first digit of } y^{*}\right] \qquad \mathrm{IIA}_{\text{whole}} = \mathbb{1}\left[\text{greedy output number} = y^{*}\right]$$

$y^{*}$ = the corrupted prompt's number at the perturbed slot.

## Setup

In [ ]:
import re
import numpy as np
import torch
import matplotlib.pyplot as plt

import data_numberreps as D          # sets HF_HOME before transformers is imported
import data_sharedrep as S
from data_numberreps import (build_triples, make_batch, regime_positions, CASE_SLOT, SEED,
                             REGIMES, EVAL_BS, N_EVAL, N_DAS, device)

TAG = '2digit'
MAX_NEW = 4        # greedy tokens per prompt: two digits, the terminator, one spare
N_SHOW = 6         # example generations printed per condition

D.load_model()
S.init_head_machinery()
tok = D.tokenizer

cfg = REGIMES[TAG]
T, (POS_A, POS_B), _, _ = regime_positions(cfg['low'], cfg['high'])
print(f'T={T}  y1 last token @ {POS_A}  y2 last token @ {POS_B}')

## Directions, triples, batches

In [ ]:
R  = np.load(f'results/numberreps_{TAG}.npz')
SR = np.load(f'results/sharedrep_{TAG}.npz')
Z  = np.load(f'results/comparator_data_{TAG}.npz')


def as_basis(v):
    v = np.asarray(v, dtype=np.float32)
    return torch.tensor(v.reshape(v.shape[0], -1), device=device)


Q_U   = as_basis(R['das_dir'])                     # u:  L13 DAS @ y1
Q_V1  = as_basis(SR['h14_das_dir'])                # v1: H14 DAS @ y2
Q_V2  = as_basis(SR['l13_pc1b_dir'])               # v2: L13 PC1 @ y2
P_UV  = as_basis(SR['uv_plane_basis'])             # span(v1, v2), orthonormal
Q_C15 = as_basis(Z['das15_basis_n1_larger_k1'])    # L15 DAS rank 1, fit with y1 perturbed

# Same pool and split as data_numberreps / data_sharedrep / data_comparator.
pool = build_triples(N_EVAL + N_DAS, SEED, cfg['low'], cfg['high'], cfg['gap'])
eval_triples = pool[:N_EVAL]
for arr in (R['triples_n1_larger'], SR['eval_triples'], Z['eval_triples']):
    assert (np.array(eval_triples) == arr).all(), 'eval triples differ from the makefig data'

DIGIT = {str(d): tok.convert_tokens_to_ids(str(d)) for d in range(10)}

CASES = {'y1': 'n1 larger', 'y2': 'n2 larger'}     # the case is the perturbation
BAT, SRC = {}, {}
for c, case in CASES.items():
    b = make_batch(eval_triples, CASE_SLOT[case], T)
    b['target'] = b['corr_pairs'][:, b['win_slot']]            # y*: what a successful patch names
    assert (np.array([DIGIT[str(v)[0]] for v in b['target']]) == b['tok_nc'].cpu().numpy()).all()
    BAT[c] = b
    SRC[c] = {l: S.clean_hidden_at(b['ids_c'], l) for l in (13, 14, 15)}
    SRC[c]['ctx'] = S.capture_ctx(b['ids_c'], POS_B)
    SRC[c]['premlp'] = S.capture_premlp(b['ids_c'], POS_B)

print(f'{N_EVAL} eval triples (a, b, c), first five: {eval_triples[:5]}')
for c, b in BAT.items():
    print(f'{c} perturbed   clean:     {tok.decode(b["ids_c"][0])!r}')
    print(f'{c} perturbed   corrupted: {tok.decode(b["ids_r"][0])!r}')

## Conditions

In [ ]:
# name: (spec for data_sharedrep.build_handles, patch position)
COND = {
    'L13 full @y1': (dict(resid=[(13, 'full', None)]), POS_A),
    'u':            (dict(resid=[(13, 'sub', Q_U)]),   POS_A),
    'L13 full @y2': (dict(resid=[(13, 'full', None)]), POS_B),
    'v2':           (dict(resid=[(13, 'sub', Q_V2)]),  POS_B),
    'v1':           (dict(head=(S.H_HEAD, Q_V1)),      POS_B),    # inside H14's output (makefig)
    'v1 pre-MLP':   (dict(premlp=Q_V1),                POS_B),    # rank 1 in the L14 pre-MLP residual
    'v1 & v2':      (dict(resid=[(13, 'sub', Q_V2)], head=(S.H_HEAD, Q_V1)), POS_B),
    'v1,v2 plane':  (dict(premlp=P_UV),                POS_B),
    'L14 full':     (dict(resid=[(14, 'full', None)]), POS_B),
    'L15 DAS':      (dict(resid=[(15, 'sub', Q_C15)]), POS_B),
}

FIG1 = [('y1', ['L13 full @y1', 'u', 'v1', 'v1 pre-MLP']), ('y2', ['L13 full @y2', 'v2'])]
FIG2 = ['L13 full @y2', 'v2', 'v1', 'v1 & v2', 'v1,v2 plane', 'L14 full']
FIG3 = ['L15 DAS']
RUNS = list(dict.fromkeys([(n, c) for c, ns in FIG1 for n in ns]
                          + [(n, c) for n in FIG2 + FIG3 for c in CASES]))

# Saved first-digit IIA per example, from the makefig data files.
SAVED = {('L13 full @y1', 'y1'): R['iia_full_n1_larger_first_number'],
         ('u', 'y1'):            R['iia_das_n1_larger_first_number'],
         ('L15 DAS', 'y1'):      Z['das15_iia_n1_larger_from_n1_larger_k1'],
         ('L15 DAS', 'y2'):      Z['das15_iia_n2_larger_from_n1_larger_k1']}
for n, key in [('L13 full @y2', 'L13_full_atn2'), ('v2', 'L13_PC1_atn2'), ('v1', 'H14_DAS'),
               ('v1 & v2', 'H14_DAS_and_L13_PC1_atn2'), ('v1,v2 plane', 'L14_uv_plane_preMLP_atn2'),
               ('L14 full', 'L14_full_atn2')]:
    for c in CASES:
        SAVED[n, c] = SR[('' if c == 'y1' else 'alt_') + 'cond_iia_' + key]
print(f'{len(RUNS)} runs:', RUNS)

## Greedy generation under a patch

In [ ]:
@torch.no_grad()
def generate(ids, handles_fn=None, n_new=MAX_NEW, bs=EVAL_BS):
    # Hooks from handles_fn(lo, hi) are live only on the prompt pass: the patch sits at its prompt
    # position and every generated token reads it through the KV cache.
    out = []
    for lo in range(0, ids.shape[0], bs):
        hi = min(lo + bs, ids.shape[0])
        handles = handles_fn(lo, hi) if handles_fn else []
        try:
            o = D.model(ids[lo:hi], use_cache=True)
        finally:
            for h in handles:
                h.remove()
        pkv, nxt = o.past_key_values, o.logits[:, -1, :].float().argmax(-1)
        toks = [nxt]
        for _ in range(n_new - 1):
            o = D.model(nxt[:, None], past_key_values=pkv, use_cache=True)
            pkv, nxt = o.past_key_values, o.logits[:, -1, :].float().argmax(-1)
            toks.append(nxt)
        out.append(torch.stack(toks, 1).cpu())
        del o, pkv
    torch.cuda.empty_cache()
    return torch.cat(out).numpy()


def score(gen, target):
    # First-digit and whole-number hits of greedy generations against `target`.
    texts = [tok.decode(g) for g in gen]
    nums = np.array([int(m.group(1)) if (m := re.match(r'\s*(\d+)', t)) else -1 for t in texts])
    first = gen[:, 0] == np.array([DIGIT[str(v)[0]] for v in target])
    return dict(gen=gen, texts=texts, nums=nums,
                first=first.astype(float), whole=(nums == target).astype(float))


def mean_se(x):
    x = np.asarray(x, dtype=float)
    return x.mean(), x.std() / np.sqrt(len(x))

## Unpatched outputs

In [ ]:
BASE = {}
for c, b in BAT.items():
    BASE[c, 'clean'] = score(generate(b['ids_c']), b['clean_pairs'].max(1))
    BASE[c, 'corr'] = score(generate(b['ids_r']), b['corr_pairs'].max(1))
    for k in ('clean', 'corr'):
        s = BASE[c, k]
        print(f'{c} perturbed, {k:>5} prompts, names the max:   first digit {s["first"].mean():.3f}   '
              f'whole number {s["whole"].mean():.3f}')

## Patched outputs

In [ ]:
RES = {}
for name, c in RUNS:
    spec, pos = COND[name]
    b, src = BAT[c], SRC[c]
    RES[name, c] = score(generate(b['ids_r'], lambda lo, hi: S.build_handles(spec, src, lo, hi, pos)),
                         b['target'])

hdr = (f'{"condition":>13} {"pert.":>5} | {"saved":>5} {"first":>5} {"agree":>5} | {"whole":>5} | '
       f'{"y* 1st digit only":>17} {"=unpatched":>10} {"other":>5}')
print(hdr + '\n' + '-' * len(hdr))
for name, c in RUNS:
    r, b = RES[name, c], BAT[c]
    sv = SAVED.get((name, c))
    fd_only = (r['first'] == 1) & (r['whole'] == 0)
    unp = r['nums'] == b['corr_pairs'].max(1)
    other = ~((r['whole'] == 1) | fd_only | unp)
    s_sv = f'{sv.mean():5.3f}' if sv is not None else '  n/a'
    s_ag = f'{(sv == r["first"]).mean():5.3f}' if sv is not None else '  n/a'
    print(f'{name:>13} {c:>5} | {s_sv} {r["first"].mean():5.3f} {s_ag} | {r["whole"].mean():5.3f} | '
          f'{fd_only.mean():17.3f} {unp.mean():10.3f} {other.mean():5.3f}')

## Example generations

In [ ]:
def row(b, c, r, i):
    cp, rp = b['clean_pairs'][i], b['corr_pairs'][i]
    return (f'  clean max({cp[0]}, {cp[1]}) -> {BASE[c, "clean"]["texts"][i]!r:8}  '
            f'corrupted max({rp[0]}, {rp[1]}) -> {BASE[c, "corr"]["texts"][i]!r:8}  '
            f'patched -> {r["texts"][i]!r:8}  y*={b["target"][i]}  '
            f'{"ok" if r["whole"][i] else "x"}')


for name, c in RUNS:
    r, b = RES[name, c], BAT[c]
    print(f'--- {name}, {c} perturbed: first digit {r["first"].mean():.3f}, whole number {r["whole"].mean():.3f}')
    for i in range(N_SHOW):
        print(row(b, c, r, i))
    bad = np.where((r['first'] == 1) & (r['whole'] == 0))[0]
    if len(bad):
        print(f'  first digit right, whole number wrong ({len(bad)} of {b["n"]}):')
        for i in bad[:N_SHOW]:
            print(row(b, c, r, i))
    print()

## First digit right, whole number wrong

Counts per patch, split by what the model says instead. Hybrid = first digit of $y^*$ + last digit of the clean (or other) number.

In [ ]:
MISS = ['hybrid: clean', 'hybrid: other', '12 (1-shot)', 'too long', 'other']


def miss_kind(num, y, clean, other):
    # What a patched output is when its first digit is y*'s but the number is not y*.
    lead = str(y)[0]
    if num == int(lead + str(clean)[-1]):
        return MISS[0]            # y*'s first digit + last digit of the clean number at that slot
    if num == int(lead + str(other)[-1]):
        return MISS[1]            # y*'s first digit + last digit of the other number
    if num == 12:
        return MISS[2]            # the answer of the one-shot example
    if len(str(num)) > len(str(y)):
        return MISS[3]
    return MISS[4]


hdr = (f'{"condition":>13} {"pert.":>5} | {"1st right":>9} {"whole wrong":>11} {"rate":>5} | '
       + ' '.join(f'{k:>13}' for k in MISS) + f' | {"hits a hybrid":>13}')
print(hdr + '\n' + '-' * len(hdr))
for name, c in RUNS:
    r, b = RES[name, c], BAT[c]
    w = b['win_slot']
    clean, other = b['clean_pairs'][:, w], b['corr_pairs'][:, 1 - w]
    first, whole = r['first'] == 1, r['whole'] == 1
    miss = np.where(first & ~whole)[0]
    kinds = [miss_kind(r['nums'][i], b['target'][i], clean[i], other[i]) for i in miss]
    # whole-number hits where the clean-hybrid equals y* anyway (clean and y* share the last digit)
    ambig = whole & np.array([str(a)[-1] == str(y)[-1] for a, y in zip(clean, b['target'])])
    rate = f'{len(miss) / first.sum():5.3f}' if first.sum() else '  n/a'
    print(f'{name:>13} {c:>5} | {first.sum():9d} {len(miss):11d} {rate} | '
          + ' '.join(f'{kinds.count(k):13d}' for k in MISS) + f' | {ambig.sum():13d}')

## IIA: $\mathbf{u}$, $\mathbf{v}_1$, $\mathbf{v}_2$

In [ ]:
COL = dict(full='#2166AC', das='#C97B00', pc1='#C06A9B', h14='#117777')
CASE_COL = {'y1': COL['das'], 'y2': COL['full']}
CASE_LAB = {'y1': '$y_1$ perturbed', 'y2': '$y_2$ perturbed'}
U, V1, V2 = r'$\mathbf{u}$', r'$\mathbf{v}_1$', r'$\mathbf{v}_2$'
METRICS = [('first', 'IIA, first digit'), ('whole', 'IIA, whole number')]
FS = dict(axis=14, tick=12, legend=12)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

BAR1 = {'L13 full @y1': ('full', COL['full'], None), 'L13 full @y2': ('full', COL['full'], None),
        'u': (U, COL['das'], None), 'v1': (V1, COL['h14'], None),
        'v1 pre-MLP': (V1 + ' pre-MLP', COL['h14'], '//'), 'v2': (V2, COL['pc1'], None)}
W1 = 0.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, (metric, ylab) in zip(axes, METRICS):
    seen = set()
    for gi, (c, names) in enumerate(FIG1):
        for k, name in enumerate(names):
            lab, colr, hatch = BAR1[name]
            m, se = mean_se(RES[name, c][metric])
            ax.bar(gi + (k - (len(names) - 1) / 2) * W1, m, W1, yerr=se, capsize=3, color=colr,
                   hatch=hatch, edgecolor='white', linewidth=0, label=None if lab in seen else lab)
            seen.add(lab)
    ax.set_xticks(range(len(FIG1)))
    ax.set_xticklabels([f'$y_{c[1]}$' for c, _ in FIG1], fontsize=FS['tick'])
    ax.set_xlabel('number perturbed', fontsize=FS['axis'])
    ax.set_ylabel(ylab, fontsize=FS['axis'])
    ax.set_ylim(0, 1.05)
    ax.tick_params(labelsize=FS['tick'])
    ax.yaxis.set_tick_params(labelleft=True)
fig.legend(*axes[0].get_legend_handles_labels(), fontsize=FS['legend'], frameon=False,
           loc='center left', bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.show()
for c, names in FIG1:
    for name in names:
        r = RES[name, c]
        print(f'{c} perturbed  {name:>13}:  first digit {r["first"].mean():.3f}   whole number {r["whole"].mean():.3f}')

## IIA: the $(\mathbf{v}_1, \mathbf{v}_2)$ space

In [ ]:
LAB2 = {'L13 full @y2': r'$\mathbf{z}^2_{l=13}$', 'v2': V2, 'v1': V1, 'v1 & v2': f'{V1}&{V2}',
        'v1,v2 plane': f'{V1},{V2}\nplane', 'L14 full': r'$\mathbf{z}^2_{l=14}$'}
W2 = 0.38
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
for ax, (metric, ylab) in zip(axes, METRICS):
    for j, c in enumerate(CASES):
        for k, name in enumerate(FIG2):
            m, se = mean_se(RES[name, c][metric])
            ax.bar(k + (j - 0.5) * W2, m, W2, yerr=se, capsize=3, color=CASE_COL[c], linewidth=0,
                   label=CASE_LAB[c] if k == 0 else None)
    ax.set_xticks(range(len(FIG2)))
    ax.set_xticklabels([LAB2[n] for n in FIG2], fontsize=FS['tick'])
    ax.set_xlabel('patched component', fontsize=FS['axis'])
    ax.set_ylabel(ylab, fontsize=FS['axis'])
    ax.set_ylim(0, 1.15)
    ax.tick_params(labelsize=FS['tick'])
    ax.yaxis.set_tick_params(labelleft=True)
axes[0].legend(fontsize=FS['legend'], frameon=False, loc='upper left', ncol=2)
fig.tight_layout()
plt.show()
for name in FIG2:
    print(f'{name:>13}: ' + '   '.join(f'{c} perturbed first {RES[name, c]["first"].mean():.3f} '
                                        f'whole {RES[name, c]["whole"].mean():.3f}' for c in CASES))

## IIA: comparator (L15 DAS, rank 1)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.2))
for j, c in enumerate(CASES):
    for k, (metric, _) in enumerate(METRICS):
        m, se = mean_se(RES['L15 DAS', c][metric])
        ax.bar(k + (j - 0.5) * W2, m, W2, yerr=se, capsize=3, color=CASE_COL[c], linewidth=0,
               label=CASE_LAB[c] if k == 0 else None)
ax.set_xticks(range(len(METRICS)))
ax.set_xticklabels(['first digit', 'whole number'], fontsize=FS['tick'])
ax.set_ylabel('IIA, DAS 1D at L15', fontsize=FS['axis'])
ax.set_ylim(0, 1.15)
ax.tick_params(labelsize=FS['tick'])
ax.legend(fontsize=FS['legend'], frameon=False, loc='upper left', ncol=2)
fig.tight_layout()
plt.show()
print('   '.join(f'{c} perturbed first {RES["L15 DAS", c]["first"].mean():.3f} '
                 f'whole {RES["L15 DAS", c]["whole"].mean():.3f}' for c in CASES))